## DSPy Ollama Llama3 Information Extraction Pydantic

#### Load in Python Libraries

In [1]:
import os 
import sys
import re
from dotenv import load_dotenv
load_dotenv()
pythonpath = os.getenv('PYTHONPATH')
if pythonpath:
    sys.path.extend(pythonpath.split(os.pathsep))

import dspy
from transformers import AutoTokenizer, AutoModelForCausalLM
from rich import print
import pandas as pd
import ast

from dspy.teleprompt import BootstrapFewShot, BootstrapFewShotWithRandomSearch
from collections.abc import Iterable


from dspy.evaluate.evaluate import Evaluate

from rouge_score import rouge_scorer
from pydantic import BaseModel

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2', 'rougeL'], use_stemmer=True)
import json

from data.train_examples import train_example_list
from data.valid_examples import dev_example_list
from data.test_example import test_examples_list

/Users/justinvhuang/miniconda3/envs/dspy/lib/python3.11/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


#### Helper Functions

In [2]:

def validate_ans(example, pred, trace = None):
    
    job_dict = {"position_title": example.position_title,
                    "location" : example.location,
                    "work_arrangement": example.work_arrangement,
                    "experience": example.experience,
                    "employment_type": example.employment_type,
                    "pay": example.pay,
                    "degree": example.degree,
                    "certifications": example.certifications,
                    "required_skills": example.required_skills,}
     

    gold = re.sub(r'\n|\s+ ', '',str(job_dict)).lower()
    print(gold)

    prediction = str(pred.info_extracted).lower()
    print(prediction)

    scores = scorer.score(gold, prediction)
    score2 = scores['rouge2'][0]
    score1 = scores['rouge1'][0]
    scoreL = scores['rougeL'][0]
    score = (0.2*score1 + 0.3*score2 + 0.5* scoreL)

    print(score)

    return score

def normalize(job_post: str) -> str:
    job_post = job_post.strip('\n')

    job_post = re.sub(r'^[^\w\s]+|[^\w\s]+$', '', job_post, flags=re.UNICODE)

    job_post = job_post.strip('\n')

    return job_post.strip().lower()

#### Load in Data

In [3]:
train_examples=pd.read_csv('~/data/Manual Labeling - Sheet1.csv')

dev_examples=pd.read_csv('~/data/50examples_for_DSPy_withJson.csv')

test_examples = pd.read_csv("~/data/Manual Labeling - Sheet2.csv", header = None)

#### Set Ollama LLM using Llama 3 from Alibaba

In [4]:
llm = dspy.OllamaLocal(model='llama3', max_tokens = 1000, temperature=0.0)
dspy.settings.configure(lm=llm)

#### Set Pydantic Class

In [5]:
class JobPostingExtractionCert(BaseModel):
    certifications : list[str] 
    
class JobPostingExtractionPay(BaseModel):
    pay : str

class JobPostingExtractionPostion(BaseModel):
    position_title: str

class JobPostingExtractionLocation(BaseModel):
    location : str

class JobPostingExtractionWorkArrange(BaseModel):
    work_arrangement: str 
    
class JobPostingExtractionExp(BaseModel):
    experience : str

class JobPostingExtractionEmpType(BaseModel):
    employment_type: str
    
class JobPostingExtractionDegree(BaseModel):
    degree : str

class JobPostingExtractionSkills(BaseModel):
    required_skills : list[str]

#### Create DSPy Signature 

In [6]:
class InfoExtractorCert(dspy.Signature):
    """Extracts single certifications and credentials from a job posting not education if nothing found put not specified. short factoids. cannot be more than 20 words or 20 characters
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_cert: JobPostingExtractionCert = dspy.OutputField(desc = "a list of strings related to certifications total should be less than 20 words or 20 characters")

class InfoExtractorPay(dspy.Signature):
    """Extracts salary or pay information from job posting if nothing found put not specified. short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_pay: JobPostingExtractionPay = dspy.OutputField(desc = "a string of salary or pay information per year or per hour has to be less than 5 words")

class InfoExtractorPostion(dspy.Signature):
    """Extracts the title or name of the position from a job posting if nothing found put not specified. short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_pos: JobPostingExtractionPostion = dspy.OutputField(desc = "3 to 5 words about the title of the job position has to be less than 5 words")


class InfoExtractorLocation(dspy.Signature):
    """Extracts where the job is located in the United States if nothing found put not specified. short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_loc: JobPostingExtractionLocation = dspy.OutputField(desc = "Where the job is located City State and ZipCode has to be less than 5 words")


class InfoExtractorWork(dspy.Signature):
    """Extracts information if the job is remote, hybrid, on-site if nothing found put not specified short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_work: JobPostingExtractionWorkArrange = dspy.OutputField(desc = "4 to 5 words on if the job is remote hybrid or onsite has to be less than 3 words")

class InfoExtractorExp(dspy.Signature):
    """Extracts information on relevant years of experience required for a job if nothing found put not specified. short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_exp: JobPostingExtractionExp = dspy.OutputField(desc = "5 to 8 words on years of experience needed has to be less than 5 words")

class InfoExtractorEmpType(dspy.Signature):
    """Extracts information if the job is full-time part-time contract internship if nothing found put not specified short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_emp: JobPostingExtractionEmpType = dspy.OutputField(desc = "5 to 8 words about the job type has to be less than 2 words")

class InfoExtractorDeg(dspy.Signature):
    """Extracts education or university information from job posting if nothing found put not specified short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_deg: JobPostingExtractionDegree = dspy.OutputField(desc = "5 to 8 words wheter its bachelors, masters, high school or PHD has to be less than 5 words")

class InfoExtractorSkills(dspy.Signature):
    """Extracts relevant skills needed to perform job should not be more than 20 words or 20 characters if nothing found put not specified short factoids. 
    """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted_skills: JobPostingExtractionSkills = dspy.OutputField(desc = "total should not be more than 20 words or 20 characters")

#### Crease DSPy Module

In [7]:
class JobPostingModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.info_extraction_modules = {
            "position_title": dspy.ChainOfThoughtWithHint(InfoExtractorPostion, max_tokens=15,),
            "location": dspy.ChainOfThoughtWithHint(InfoExtractorLocation, max_tokens=15),
            "work_arrangement": dspy.ChainOfThoughtWithHint(InfoExtractorWork, max_tokens=30),
            "experience": dspy.ChainOfThoughtWithHint(InfoExtractorExp, max_tokens=15),
            "employment_type": dspy.ChainOfThoughtWithHint(InfoExtractorEmpType, max_tokens=15),
            "pay": dspy.ChainOfThoughtWithHint(InfoExtractorPay, max_tokens=15),
            "degree": dspy.ChainOfThoughtWithHint(InfoExtractorDeg, max_tokens=15),
            "certifications": dspy.ChainOfThoughtWithHint(InfoExtractorCert, max_tokens=30),
            "required_skills": dspy.ChainOfThoughtWithHint(InfoExtractorSkills, max_tokens=30)
        }

        self.attribute_names = {
            "position_title": "info_extracted_pos",
            "location": "info_extracted_loc",
            "work_arrangement": "info_extracted_work",
            "experience": "info_extracted_exp",
            "employment_type": "info_extracted_emp",
            "pay": "info_extracted_pay",
            "degree": "info_extracted_deg",
            "certifications": "info_extracted_cert",
            "required_skills": "info_extracted_skills"
        }

        self.hints = {
            "certifications": "Usually 3 or 4 letters capitalized, not an academic degree like BS, Masters or PHD but something you obtain professionally or through job experience or relevant technologies or certificates",
            "pay": "The dollars per hour or salary for the year or pay",
            "position_title": "The title of the job or position of the job",
            "location": "Where the job is located in the United States",
            "work_arrangement": "If the job is hybrid, remote on-site or where the job is located to go into the office",
            "experience": "Number of years of experience that a job posting is asking for",
            "employment_type": "full time, part time, contract or internship or apprenticeship",
            "degree": "Education level such as GED, High School, Bachelors, Masters, Doctorate, PHD",
            "required_skills": "The required skills to do the job"
        }

    def forward(self, job_posting):
        job_posting = job_posting.replace('\n', ' ').replace('“', '"').replace('”', '"')
        job_posting = normalize(job_posting)

        job_dict = {}
        for key in ["position_title", "location", "work_arrangement", "experience", "employment_type", "pay", "degree", "certifications", "required_skills"]:
            extraction_module = self.info_extraction_modules[key]
            hint_value = self.hints[key]
            extracted_info = getattr(extraction_module(job_posting=job_posting, hint=hint_value), self.attribute_names[key]).replace("```\n", "").replace("```", "")
            dspy.Suggest(len(extracted_info) <= 400,f"info extract should be short and less than 400 characters right now its {len(extracted_info)} for {key}",)
            job_dict[key] = extracted_info

        return dspy.Prediction(job_posting=job_posting, info_extracted=job_dict)


In [8]:
uncompiled_module = JobPostingModule()

#### Perform Test Extraction on Data with uncompiled version

In [9]:
print(test_examples[1][5])


EMT-Advanced-Emergency Medical Service
Job Locations
US-TX-Rosenberg
Posted Date
9 months ago
(5/16/2023 11:01 AM)
ID 2023-5265 Job Code DJOB # of Openings 10 Min Start Salary USD $2,008.28/Bi. Category EMS Max Start Salary USD 
$2,421.41/Bi.
Overview

Fort Bend County is ranked as one of the fastest growing counties in the nation. We have capitalized on not only 
the creed of our location, but on the "quality of life" for our families to call home. Our employees are the key to
our success and the heartbeat of our foundation. The diversity and inclusivity of our community is our strength and
at the forefront of a workplace environment welcoming to all. Live Here! Work Here!



Provides emergency medical care to the citizens of Fort Bend County as stated in established standards and 
procedures.



Responsibilities
Provides emergency pre-hospital medical care. 
Completes reports within established timeframes.
Maintains emergency vehicle(s) and inventory of medical supplies.
Signs for and is held accountable for equipment issued and used.
Operates emergency vehicles (i.e. ambulance, squad) per department policy and with due regard to the law.
Responsible for maintaining all current certifications within department guidelines as required.
Certifications shall include EMT-Basic that has successfully completed AEMT and is test eligible and is enrolled in
an EMT Paramedic Program, DSHS EMT-Advanced Certification who is enrolled in an EMT Paramedic Program, and Valid 
State of Texas Driver's License.
Prepares, submits, and maintains clear, concise, and accurate documentation on patient care activities, incident 
reports and other related information as requested.
Assists other employees with their duties.
Performs general housekeeping duties for station and department areas.
Participates in activities and duties related to emergency management during a local state of disaster as directed 
by appropriate county managers.
Qualifications
High School Diploma/GED; Enrolled in College pursing Paramedic Certification and/or EMS Degree.
Certified or Licensed State of Texas EMT-Basic or EMT-Advanced or is eligible to test for EMT-Advanced 
Certification. 
Current Healthcare Provider CPR/AED card.
Pre-hospital experience preferred.
Experience in a high performance ALS system beneficial.
Strong verbal and written communication and organizational skills.
Strong interpersonal skills and ability to deal effectively with the public and other employees. 
Frequent reading, writing, memorization, analyzing, simple math skills, negotiating. Constantly using judgment, 
reasoning, decision-making and teaching.
Ability to complete projects.
Must obtain and maintain a current American Heart Association Advanced Cardiac Life Support certification.
Must complete National Incident Management System (NIMS) 100, 200, 700 and 800 within 90 days of hire.
Must obtain Paramedic Credentials in 24 months after hire date. Subject to emergency call-in and mandatory 
staffing.



SALARY RANGE: EMS Grade EMT-1, $2,008.28 - $2,421.41 biweekly based on qualifications

CLOSING DATE: Upon filling position





Fort Bend County is an equal opportunity employer, committed to non-discrimination in employment on any basis 
including race, color, religion or creed, sex, sexual orientation, gender, gender identity, gender expression, 
pregnancy status (including childbirth and related medical conditions), national origin, ethnicity, citizenship 
status, age (40 and over), physical or mental disability, genetic information, protected military and veteran 
status, political affiliation or beliefs, or any other classification protected by state, federal and local laws, 
unless such classification is a bona fide occupational qualification.

In [10]:
with dspy.context(lm = llm):
    pred = uncompiled_module(job_posting =test_examples[1][5])
    print(pred.info_extracted)

{
    'position_title': 'Emergency Medical Technician',
    'location': 'Rosenberg TX',
    'work_arrangement': 'Not Specified',
    'experience': 'Not Specified',
    'employment_type': 'Full-time',
    'pay': '$2,008.28',
    'degree': 'High School Diploma/GED\n\n---\n\nNote: The job posting specifies that the required education level 
is a "high school diploma/ged; enrolled in college pursuing paramedic certification and/or EMS degree."',
    'certifications': '* CPR/AED\n* NIMS (100, 200, 700, and 800)\n* AHA Advanced Cardiac Life Support 
Certification',
    'required_skills': 'strong verbal and written communication, organizational, interpersonal, frequent reading, 
writing, memorization, analyzing, simple math skills, negotiating, judgment, reasoning, decision-making, teaching.'
}

#### Inspect History and save DSPy program 

In [11]:
#print(llm.inspect_history(n=1))
uncompiled_module.save("uncompiled_file_pydantic_v2.json")

#### Create Training Examples, Validation(Dev) and Test Examples

In [12]:
print(len(dev_example_list) , len(train_example_list), len(test_examples_list))
train_results = train_example_list
train_contents = list(train_examples['body'])

dev_results = dev_example_list
dev_contents = list(dev_examples.loc[:20,'body'])

test_results = test_examples_list
test_contents = list(test_examples[1])

21 20 10

In [13]:
train_examples_list = [
    dspy.Example(
        job_posting=content,
        position_title=str(pos['position_title']),
        location=str(loc['location']),
        work_arrangement=str(work['work_arrangement']),
        experience=str(exp['experience']),
        employment_type=str(emp['employment_type']),
        pay=str(pay['pay']),
        degree=str(deg['degree']),
        certifications=str(cer['certification']),
        required_skills=str(req['required_skills'])
    )
    for content, pos, loc, work, exp, emp, pay, deg, cer, req in zip(
        train_contents, *([[json.loads(x) for x in train_results]] * 9)
    )
]

dev_examples_list = [
    dspy.Example(
        job_posting=content,
        position_title=str(pos['position_title']),
        location=str(loc['location']),
        work_arrangement=str(work['work_arrangement']),
        experience=str(exp['experience']),
        employment_type=str(emp['employment_type']),
        pay=str(pay['pay']),
        degree=str(deg['degree']),
        certifications=str(cer['certification']),
        required_skills=str(req['required_skills'])
    )
    for content, pos, loc, work, exp, emp, pay, deg, cer, req in zip(
        dev_contents, *([[json.loads(x) for x in dev_results]] * 9)
    )
]

test_examples_list = [
    dspy.Example(
        job_posting=content,
        position_title=str(pos['position_title']),
        location=str(loc['location']),
        work_arrangement=str(work['work_arrangement']),
        experience=str(exp['experience']),
        employment_type=str(emp['employment_type']),
        pay=str(pay['pay']),
        degree=str(deg['degree']),
        certifications=str(cer['certification']),
        required_skills=str(req['required_skills'])
    )
    for content, pos, loc, work, exp, emp, pay, deg, cer, req in zip(
        test_contents, *([[json.loads(x) for x in test_results]] * 9)
    )
]

In [14]:
trainset=train_examples_list
devset=dev_examples_list
testset = test_examples_list

trainset = [x.with_inputs('job_posting') for x in trainset]
devset = [x.with_inputs('job_posting') for x in devset]
testset = [x.with_inputs('job_posting') for x in testset]

In [15]:
print(train_examples['body'][0])

Derrickhand, Buckhannon, WV


Job Order Number
        
WV2925359


Post Date
        
05/12/2023


Job Location
        
Buckhannon, West Virginia 26201


County
        
Upshur


Job Summary
        
Job Summary: This position is a crew member assigned to work on a well service rig, responsible for performing 
services on oil and gas wells. Duties include performing all well-servicing tasks from an elevated position (rod 
basket or tubing board), assisting in rigging up or down, picking up or laying down tubing, and other functions 
specified by the customer or well operator. This position has a dotted line reporting line to: Rig Supervisor. 
Responsibilities: Assists the operator in rigging up and down, lining up the well service rig with the well. Sets 
hydraulic jacks, handles pads/boards and assists in attaching the guy wires to the anchor. Responsible for all 
elevated work associated with rigging up/down (i.e. removing horse head from pumping unit). Responsible for all 
work performed for the rod basket and tubing board (transferring rods and tubing from the vertical racks to the 
elevator), performs servicing on the well. Drives the crew truck as needed. Operates tubing elevators for standing 
tubing in derrick. Assists in picking up or laying down tubing, manually lifting the tubing from the rack onto the 
work floor or vice versa. Assists in walking the rods when laying down rods. Reports any safety hazards, accidents 
or maintenance issues to the rig supervisor. Ensures that work carried out is in compliance with company policies 
and procedures and according to safety regulations. May be required to work floors or operate the rig when needed. 
Performs other related duties as assigned. Preferred Qualifications: 1-2 years of Workover - Derrickhand experience
required. Ability to effectively communicate, both verbally and written. Ability to interact with others in a team 
environment. Ability to work in a fast-paced environment and handle multiple tasks at once. Basic problem solving 
and organizational skills. Excellent customer service skills, to provide world class value to customers CDL B 
license is required to drive rig. Must meet all qualifications defined in the Motor Vehicle Policy if required to 
drive. Ability to communicate verbally and in writing, in English, is preferred. Education Requirements: High 
school diploma, GED, or the equivalent is preferred. We are proud to offer a very competitive compensation and 
benefits package including: Medical Insurance Vision and Dental Insurance Life Insurance 401(k) Education 
assistance Short-Term Disability Paid time-off Request Priority Protected Veteran Referrals Equal Opportunity 
Employer - minorities/females/veterans/individuals with disabilities/sexual orientation/gender identity


Experience
        
0 Months


Pay Rate
        
0 $ / Hour


Master Group
        
Construction and Extraction Occupations


Job Type
        
Rotary Drill Operators, Oil and Gas


Shift
        
Day Shift

#### Test Uncompiled Module on combined Rouge Score

In [16]:
answ = trainset[0]
print(answ)

Example({'job_posting': 'Derrickhand, Buckhannon, WV\n\n\nJob Order Number\n        \nWV2925359\n\n\nPost Date\n   
\n05/12/2023\n\n\nJob Location\n        \nBuckhannon, West Virginia 26201\n\n\nCounty\n        \nUpshur\n\n\nJob 
Summary\n        \nJob Summary: This position is a crew member assigned to work on a well service rig, responsible 
for performing services on oil and gas wells. Duties include performing all well-servicing tasks from an elevated 
position (rod basket or tubing board), assisting in rigging up or down, picking up or laying down tubing, and other
functions specified by the customer or well operator. This position has a dotted line reporting line to: Rig 
Supervisor. Responsibilities: Assists the operator in rigging up and down, lining up the well service rig with the 
well. Sets hydraulic jacks, handles pads/boards and assists in attaching the guy wires to the anchor. Responsible 
for all elevated work associated with rigging up/down (i.e. removing horse head from pumping unit). Responsible for
all work performed for the rod basket and tubing board (transferring rods and tubing from the vertical racks to the
elevator), performs servicing on the well. Drives the crew truck as needed. Operates tubing elevators for standing 
tubing in derrick. Assists in picking up or laying down tubing, manually lifting the tubing from the rack onto the 
work floor or vice versa. Assists in walking the rods when laying down rods. Reports any safety hazards, accidents 
or maintenance issues to the rig supervisor. Ensures that work carried out is in compliance with company policies 
and procedures and according to safety regulations. May be required to work floors or operate the rig when needed. 
Performs other related duties as assigned. Preferred Qualifications: 1-2 years of Workover - Derrickhand experience
required. Ability to effectively communicate, both verbally and written. Ability to interact with others in a team 
environment. Ability to work in a fast-paced environment and handle multiple tasks at once. Basic problem solving 
and organizational skills. Excellent customer service skills, to provide world class value to customers CDL B 
license is required to drive rig. Must meet all qualifications defined in the Motor Vehicle Policy if required to 
drive. Ability to communicate verbally and in writing, in English, is preferred. Education Requirements: High 
school diploma, GED, or the equivalent is preferred. We are proud to offer a very competitive compensation and 
benefits package including: Medical Insurance Vision and Dental Insurance Life Insurance 401(k) Education 
assistance Short-Term Disability Paid time-off Request Priority Protected Veteran Referrals Equal Opportunity 
Employer - minorities/females/veterans/individuals with disabilities/sexual orientation/gender 
identity\n\n\nExperience\n        \n0 Months\n\n\nPay Rate\n        \n0 $ / Hour\n\n\nMaster Group\n        
\nConstruction and Extraction Occupations\n\n\nJob Type\n        \nRotary Drill Operators, Oil and Gas\n\n\nShift\n
\nDay Shift', 'position_title': 'Derrickhand', 'location': 'Buckhannon, West Virginia 26201', 'work_arrangement': 
'On-Site, Shifts', 'experience': '1-2 years of Derrickhand experience', 'employment_type': 'Full-time', 'pay': 'Not
specified', 'degree': 'High school diploma/GED or equivalent', 'certifications': 'CDL B License', 
'required_skills': 'Effective verbal/written communication in English, ability to interact with teams in a 
fast-paced environment, ability to multi-task, basic problem solving, organizational skills, excellent 
customer-service'}) (input_keys={'job_posting'})

In [17]:
with dspy.context(lm=llm):
    pred = uncompiled_module(trainset[0].job_posting)
    print(pred.info_extracted)

{
    'position_title': 'Derrickhand',
    'location': "Here's the extracted location information:\n\nJob Posting: derrickhand, buckhannon, wv job order 
number wv2925359 post date 05/12/2023 job location **Buckhannon, WV 26201**\n\nReasoning: Let's think step by step 
in order to extract the location. We know that the job is located in West Virginia (WV) and has a zip code of 
26201.\n\nInfo Extracted Loc: Buckhannon, WV",
    'work_arrangement': 'Not Specified',
    'experience': '1-2',
    'employment_type': 'Full-time',
    'pay': 'Not Specified',
    'degree': 'High School',
    'certifications': "* CDL (Commercial Driver's License)\n* OSHA (Occupational Safety and Health Administration) 
certification\n* CPR (Cardiopulmonary Resuscitation) certification",
    'required_skills': "**Job Posting:** derrickhand, buckhannon, wv job order number wv2925359 post date 
05/12/2023 job location buckhannon, west virginia 26201 county upshur\n\n**Reasoning:** Let's think step by step in
order to **Info Extracted Skills**. We ...\n\n**Hint:** The required skills to do the job\n\n**Info Extracted 
Skills:** Communication, Problem Solving, Organization, Customer Service, Teamwork"
}

In [18]:
validate_ans(answ, pred)

{'position_title': 'derrickhand', 'location': 'buckhannon, west virginia 26201', 'work_arrangement': 'on-site, 
shifts', 'experience': '1-2 years of derrickhand experience', 'employment_type': 'full-time', 'pay': 'not 
specified', 'degree': 'high school diploma/ged or equivalent', 'certifications': 'cdl b license', 
'required_skills': 'effective verbal/written communication in english, ability to interact with teams in a 
fast-paced environment, ability to multi-task, basic problem solving, organizational skills, excellent 
customer-service'}

{'position_title': 'derrickhand', 'location': "here's the extracted location information:\n\njob posting: 
derrickhand, buckhannon, wv job order number wv2925359 post date 05/12/2023 job location **buckhannon, wv 
26201**\n\nreasoning: let's think step by step in order to extract the location. we know that the job is located in
west virginia (wv) and has a zip code of 26201.\n\ninfo extracted loc: buckhannon, wv", 'work_arrangement': 'not 
specified', 'experience': '1-2', 'employment_type': 'full-time', 'pay': 'not specified', 'degree': 'high school', 
'certifications': "* cdl (commercial driver's license)\n* osha (occupational safety and health administration) 
certification\n* cpr (cardiopulmonary resuscitation) certification", 'required_skills': "**job posting:** 
derrickhand, buckhannon, wv job order number wv2925359 post date 05/12/2023 job location buckhannon, west virginia 
26201 county upshur\n\n**reasoning:** let's think step by step in order to **info extracted skills**. we 
...\n\n**hint:** the required skills to do the job\n\n**info extracted skills:** communication, problem solving, 
organization, customer service, teamwork"}

0.19663745892661555

0.19663745892661555

#### BootStrap Few Shot With A Few Examples to Change the Output
    * Note for some reason compiling does an even worse job

In [19]:
teleprompter = BootstrapFewShot(metric=validate_ans) 
compiled = teleprompter.compile(uncompiled_module, trainset=trainset)
# teleprompter = BootstrapFewShotWithRandomSearch(metric=validate_ans, max_labeled_demos=16, max_rounds=1,  max_errors = 5, stop_at_score=0.50) 
# compiled = teleprompter.compile(uncompiled_module, trainset=trainset, valset = devset)

  0%|          | 0/20 [00:00<?, ?it/s]

{'position_title': 'derrickhand', 'location': 'buckhannon, west virginia 26201', 'work_arrangement': 'on-site, 
shifts', 'experience': '1-2 years of derrickhand experience', 'employment_type': 'full-time', 'pay': 'not 
specified', 'degree': 'high school diploma/ged or equivalent', 'certifications': 'cdl b license', 
'required_skills': 'effective verbal/written communication in english, ability to interact with teams in a 
fast-paced environment, ability to multi-task, basic problem solving, organizational skills, excellent 
customer-service'}

{'position_title': 'derrickhand', 'location': "here's the extracted location information:\n\njob posting: 
derrickhand, buckhannon, wv job order number wv2925359 post date 05/12/2023 job location **buckhannon, wv 
26201**\n\nreasoning: let's think step by step in order to extract the location. we know that the job is located in
west virginia (wv) and has a zip code of 26201.\n\ninfo extracted loc: buckhannon, wv", 'work_arrangement': 'not 
specified', 'experience': '1-2', 'employment_type': 'full-time', 'pay': 'not specified', 'degree': 'high school', 
'certifications': "* cdl (commercial driver's license)\n* osha (occupational safety and health administration) 
certification\n* cpr (cardiopulmonary resuscitation) certification", 'required_skills': "**job posting:** 
derrickhand, buckhannon, wv job order number wv2925359 post date 05/12/2023 job location buckhannon, west virginia 
26201 county upshur\n\n**reasoning:** let's think step by step in order to **info extracted skills**. we 
...\n\n**hint:** the required skills to do the job\n\n**info extracted skills:** communication, problem solving, 
organization, customer service, teamwork"}

0.19663745892661555

 20%|██        | 4/20 [02:40<10:36, 39.78s/it]

...

{'position_title': 'academic instructor', 'location': 'fullerton, ca', 'work_arrangement': 'not specified', 
'experience': 'experience working with low-income and diverse student population, experience working with auhsd 
student, experience working with middle or high school students', 'employment_type': 'part-time', 'pay': 
'$47-$52/hr', 'degree': "bachelor's degree, master's degree", 'certifications': 'not specified', 'required_skills':
'teaching, ability to work in a collaborative team environment, develop effective teaching strategies, lifting of 
up to 25lbs'}

{'position_title': 'not specified', 'location': 'fullerton, ca', 'work_arrangement': "not specified\n\nsince there 
is no mention of remote, hybrid, or on-site work in the job posting, i couldn't extract any specific information 
about the work arrangement.", 'experience': 'not specified', 'employment_type': 'part-time', 'pay': '$47-$52 an 
hour', 'degree': 'minimum bachelors degree', 'certifications': '* cpr\n* e-verify', 'required_skills': 'teaching 
experience, curriculum development, lesson planning, classroom management, communication, teamwork, and 
adaptability.'}

0.34829692706405035

 35%|███▌      | 7/20 [05:15<09:46, 45.08s/it]


DSPySuggestionError: info extract should be short and less than 400 characters right now its 486 for pay

#### Test Compiled Version

    * Issue with Compiler adds more verbose detail for some reason making it worst than out of the box

In [21]:
# with dspy.context(lm=llm):
#     pred = compiled(job_posting=trainset[0].job_posting)
#     print(pred.info_extracted)

In [22]:
#validate_ans(answ, pred)

In [83]:
compiled.save("compiled_v2_pydantic_v2.json")

#### Do Side by Side Comparieson on Evaluation versus uncompiled vs compiled
    * Note Library not fully developed something causing it to collect more information shows on results for uncompiled version

In [23]:
evaluation = Evaluate(devset=testset, num_threads=1, display_progress=True, display_table=10,return_outputs=True)

prev_score=evaluation(uncompiled_module, metric=validate_ans)

#improved_score=evaluation(compiled, metric=validate_ans)

  0%|          | 0/10 [00:00<?, ?it/s]

{'position_title': 'senior inside sales rep/sales engineer', 'location': 'walpole, ma', 'work_arrangement': 
'hybrid', 'experience': 'depends on experience', 'employment_type': 'full time', 'pay': '$120k/year', 'degree': 'a 
bs in the engineering field', 'certifications': 'crm (salesforce.com), rfq experience and price quotes to the dod',
'required_skills': 'inside/outside technical sales experience, experience working with outside sales reps'}

{'position_title': 'senior inside sales rep/sales engineer', 'location': 'walpole, ma', 'work_arrangement': 
'hybrid', 'experience': 'not specified', 'employment_type': 'full-time', 'pay': '$120k/year', 'degree': 'not 
specified', 'certifications': 'crm', 'required_skills': 'technical sales experience, crm (salesforce.com) 
experience, rfq experience, price quotes to the dod, erp (epicor) knowledge, power electronics knowledge, sales 
experience in military and industrial sectors.'}

0.6304900181488203

Average Metric: 0.6304900181488203 / 1  (63.0):  10%|█         | 1/10 [01:53<09:12, 61.41s/it]

...

{'position_title': 'emt-advanced-emergency medical service', 'location': 'rosenberg, tx 77471', 'work_arrangement':
'on-site', 'experience': 'pre-hospital experience preferred, experience in a high performance als system', 
'employment_type': 'full time', 'pay': '$2,008.28 - $2,421.41 biweekly', 'degree': 'high school diploma/ged', 
'certifications': "paramedic certification or ems degree, aemt, enrolled in an emt paramedic program, dshs 
emt-advanced, valid texas driver's license", 'required_skills': 'strong verbal and written communication, 
organizational skills, interpersonal skills, judgment, reasoning, decision-making, teaching'}

{'position_title': 'emergency medical technician', 'location': 'rosenberg tx', 'work_arrangement': 'not specified',
'experience': 'not specified', 'employment_type': 'full-time', 'pay': '$2,008.28', 'degree': 'high school 
diploma/ged\n\n---\n\nnote: the job posting specifies that the required education level is a "high school 
diploma/ged; enrolled in college pursuing paramedic certification and/or ems degree."', 'certifications': '* 
cpr/aed\n* nims (100, 200, 700, and 800)\n* aha advanced cardiac life support certification', 'required_skills': 
'strong verbal and written communication, organizational, interpersonal, frequent reading, writing, memorization, 
analyzing, simple math skills, negotiating, judgment, reasoning, decision-making, teaching.'}

0.4278350515463918

Average Metric: 1.0583250696952122 / 6  (17.6):  60%|██████    | 6/10 [03:54<02:35, 38.84s/it]

DSPySuggestionError: info extract should be short and less than 400 characters right now its 1325 for degree